In [1]:

#load the packages
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, train_test_split
import os
import pandas as pd
import html

In [2]:
#set working directory
os.chdir('/Users/fan/Downloads/ling-539-competition-2026')

In [3]:
# load the datasets
train= pd.read_csv("train.csv")
test= pd.read_csv("test.csv")

In [4]:
#check both datasets 
print(train.head(5))
print(test.head(5))

             ID                                               TEXT  LABEL
0  3.279960e+17  Carolyn Baker defines the niche of helping peo...      0
1  1.184830e+19  Just getting released from a six month drug re...      1
2  8.453490e+18  I was greatly dissappointed when I popped this...      0
3  1.308890e+19  This is a film that has garnered any interest ...      2
4  4.199320e+18  This is one of the more adorable episodes of t...      1
                     ID                                               TEXT
0   1087873697464833975  This tv series is one of the best I have seen ...
1   5853461517618045821  I saw Evanescence live this summer and it was ...
2   1225516091890726770  Almost from the word go this film is poor and ...
3  11795874926011119457  I bought this book and I know friends who have...
4  15956464546963841646  “As you please.”\n\n“Ah, I forgot! A letter ca...


In [5]:
#check dimension
print(train.shape)
print(test.shape)

(70317, 3)
(17580, 2)


In [6]:
#dataset size (not a review)
(train['LABEL'] == 0).sum()

32289

In [7]:
#dataset size (positive)
(train['LABEL'] == 1).sum()

19139

In [8]:
#dataset size (negative)
(train['LABEL'] == 2).sum()

18889

In [9]:
#divide the datasets to feature and outcome
X_train = train["TEXT"]
y_train = train["LABEL"]
X_test = test["TEXT"]  #Test has no label column

In [10]:
type(y_train)

pandas.core.series.Series

In [11]:
#missing values
X_train = X_train.fillna("")
X_test = X_test.fillna("")

In [12]:
#remove meaningless symbols and whitespaces
X_train = X_train.astype(str).map(html.unescape).str.replace(r"<br\s*/?>", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
X_test = X_test.astype(str).map(html.unescape).str.replace(r"<br\s*/?>", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()


In [13]:
#print(X_train.head(10))

In [14]:
#classification model
classifier = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.90,
        sublinear_tf=True,
        max_features=100000
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        C=4,
        class_weight="balanced",
        solver="saga",
        random_state=42
    ))
])

In [15]:
#fit the model
classifier.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, max_features=100000, min_df=2,
                                 ngram_range=(1, 2), strip_accents='unicode',
                                 sublinear_tf=True)),
                ('clf',
                 LogisticRegression(C=4, class_weight='balanced', max_iter=3000,
                                    random_state=42, solver='saga'))])

In [16]:
#making predictions using the test dataset
predictions = classifier.predict(X_test)

In [17]:
#output to the submission csv file
results = pd.DataFrame({
    "ID": test["ID"],  
    "LABEL": predictions
})

# Save
results.to_csv("predictions.csv", index=False)

In [18]:
#Measure of robustness

#StratifiedKFold and Cross Validation
stratifiedKFold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
cv_results = cross_validate(
    classifier,
    X_train,
    y_train,
    cv=stratifiedKFold,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro"},
    return_train_score=True,
    n_jobs=-1
)
print("Accuracy:", cv_results["test_accuracy"].mean())
print("Macro F1:", cv_results["test_macro_f1"].mean())

Accuracy: 0.9331598645927552
Macro F1: 0.9233553605924273


In [19]:
#Error analysis on held-out data
# We don't have true labels for the test dataset, so we split the training dataset.
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

classifier.fit(X_train_new, y_train_new)

y_test_new_pred = classifier.predict(X_test_new)

output = pd.DataFrame({
    "TEXT": X_test_new,
    "TRUE_LABEL": y_test_new,
    "PREDICTED_LABEL": y_test_new_pred
})

error_cases = output[output["TRUE_LABEL"] != output["PREDICTED_LABEL"]]


In [20]:
print(error_cases.head(10))

                                                    TEXT  TRUE_LABEL  \
70195  The 1930' were a golden age of Los Angeles wit...           2   
37901  I saw this film as a sneak preview before the ...           2   
58774  This is the best movie I`ve ever seen !!! Thom...           1   
24269  like i'm sure other people have said this guy ...           2   
64138  This is a slick little movie well worth your t...           2   
60913  A breathtaking view of isolation and lonelines...           1   
7577   This game is full of levels, characters, and h...           0   
63215  Lots of scenes and dialogue are flat-out goofy...           1   
9293   I watch a lot of documentaries on health, fitn...           1   
29810  To all the haters out there: condemning a TV s...           1   

       PREDICTED_LABEL  
70195                1  
37901                1  
58774                2  
24269                0  
64138                1  
60913                0  
7577                 1  
63215  

In [21]:
error_cases.to_csv('error cases.csv')